In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🎫 Escalation-Aware Support Workflow

## What We're Building

A customer support system that **intelligently routes tickets**:
- **Simple tickets** → auto-resolved by LLM (password resets, FAQs)
- **Medium tickets** → LLM drafts response, auto-sends with logging
- **Hard tickets** → escalated to human agent for manual review

## Architecture

```
Incoming Ticket
      │
      ▼
┌──────────────┐
│  Classify     │  → Determine difficulty + category
└──────┬───────┘
       │
  ┌────┼────────────┐
  │    │             │
  ▼    ▼             ▼
EASY  MEDIUM        HARD
  │    │             │
  ▼    ▼             ▼
Auto  Draft+Send   Escalate
Reply  Response     to Human
```

## Key LangGraph Concept: Conditional Routing

Use `add_conditional_edges()` to route to different nodes based on
the classification result. Each branch handles the ticket differently.

## Stack
- **LangGraph** — routing nodes + conditional edges
- **Ollama** — local LLM

## Step 1 — Imports & Setup

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)
print("All imports successful!")

## Step 2 — Define State & Sample Tickets

In [ ]:
class SupportState(TypedDict):
    ticket_id: str
    customer_name: str
    ticket_text: str
    category: str           # billing, technical, account, etc.
    difficulty: str         # easy, medium, hard
    sentiment: str          # positive, neutral, frustrated, angry
    response: str           # Generated response
    resolution_path: str    # auto_resolved, drafted, escalated
    escalation_reason: str  # Why it was escalated (if applicable)


# Sample tickets with varying difficulty
tickets = [
    {
        "ticket_id": "T-1001",
        "customer_name": "Sarah Chen",
        "ticket_text": "How do I reset my password? I forgot it and can't log in."
    },
    {
        "ticket_id": "T-1002",
        "customer_name": "Mike Torres",
        "ticket_text": "I was charged twice for my subscription this month. Transaction IDs: TXN-4421 and TXN-4422. Both for $49.99 on March 15. Please refund the duplicate charge."
    },
    {
        "ticket_id": "T-1003",
        "customer_name": "Emma Kowalski",
        "ticket_text": "Your API has been returning 502 errors intermittently for the past 3 hours. This is impacting our production system serving 50,000 users. We need immediate resolution or we'll have to switch providers. Our SLA guarantees 99.99% uptime. Case ref: INC-2024-0892."
    },
    {
        "ticket_id": "T-1004",
        "customer_name": "James Wilson",
        "ticket_text": "What are your business hours? I need to call about my account."
    },
    {
        "ticket_id": "T-1005",
        "customer_name": "Priya Sharma",
        "ticket_text": "URGENT: Our enterprise account data appears to have been accessed by an unauthorized IP address (192.168.x.x from an unknown location). We see login events at 3AM that none of our team made. This could be a data breach. We need your security team involved NOW."
    },
]

print(f"Loaded {len(tickets)} sample tickets")

## Step 3 — Build Graph Nodes

In [ ]:
# --- Node 1: Classify the ticket ---
classify_prompt = ChatPromptTemplate.from_template(
    """Classify this support ticket.

Ticket: {ticket_text}

Respond in this EXACT format (one per line):
CATEGORY: [billing/technical/account/security/general]
DIFFICULTY: [easy/medium/hard]
SENTIMENT: [positive/neutral/frustrated/angry]

Rules:
- EASY: simple questions, password resets, FAQ-type queries
- MEDIUM: billing disputes, feature requests, moderate technical issues
- HARD: security incidents, SLA violations, data breaches, legal threats, production outages

Classification:"""
)


def classify_ticket(state: SupportState) -> dict:
    print(f"  🏷️ Classifying ticket {state['ticket_id']}...")
    chain = classify_prompt | llm | StrOutputParser()
    result = chain.invoke({"ticket_text": state["ticket_text"]})

    # Parse classification
    category = "general"
    difficulty = "medium"
    sentiment = "neutral"
    for line in result.strip().split("\n"):
        line_upper = line.upper()
        if "CATEGORY:" in line_upper:
            category = line.split(":", 1)[1].strip().lower()
        elif "DIFFICULTY:" in line_upper:
            difficulty = line.split(":", 1)[1].strip().lower()
        elif "SENTIMENT:" in line_upper:
            sentiment = line.split(":", 1)[1].strip().lower()

    print(f"     → category={category}, difficulty={difficulty}, sentiment={sentiment}")
    return {"category": category, "difficulty": difficulty, "sentiment": sentiment}


# --- Node 2a: Auto-resolve easy tickets ---
auto_prompt = ChatPromptTemplate.from_template(
    """Generate a helpful, friendly response to this simple support question.
Be concise and provide step-by-step instructions if applicable.

Customer: {customer_name}
Question: {ticket_text}

Response:"""
)


def auto_resolve(state: SupportState) -> dict:
    print(f"  ✅ Auto-resolving ticket {state['ticket_id']} (easy)")
    chain = auto_prompt | llm | StrOutputParser()
    response = chain.invoke({
        "customer_name": state["customer_name"],
        "ticket_text": state["ticket_text"]
    })
    return {"response": response, "resolution_path": "auto_resolved"}


# --- Node 2b: Draft response for medium tickets ---
draft_prompt = ChatPromptTemplate.from_template(
    """Draft a professional response to this support ticket.
Acknowledge the issue, explain what you'll do to help, and set expectations.

Customer: {customer_name}
Category: {category}
Ticket: {ticket_text}

Draft response:"""
)


def draft_response(state: SupportState) -> dict:
    print(f"  📝 Drafting response for ticket {state['ticket_id']} (medium)")
    chain = draft_prompt | llm | StrOutputParser()
    response = chain.invoke({
        "customer_name": state["customer_name"],
        "category": state["category"],
        "ticket_text": state["ticket_text"]
    })
    return {"response": response, "resolution_path": "drafted"}


# --- Node 2c: Escalate hard tickets ---
escalate_prompt = ChatPromptTemplate.from_template(
    """This ticket needs human escalation. Write a brief internal escalation note.

Include:
- Why this needs a human agent
- Key facts from the ticket
- Suggested priority level (P1/P2/P3)
- Recommended team (security/engineering/management/billing)

Customer: {customer_name}
Category: {category}
Sentiment: {sentiment}
Ticket: {ticket_text}

Escalation note:"""
)


def escalate_to_human(state: SupportState) -> dict:
    print(f"  🚨 Escalating ticket {state['ticket_id']} (hard) to human agent")
    chain = escalate_prompt | llm | StrOutputParser()
    reason = chain.invoke({
        "customer_name": state["customer_name"],
        "category": state["category"],
        "sentiment": state["sentiment"],
        "ticket_text": state["ticket_text"]
    })
    return {
        "response": f"[ESCALATED TO HUMAN AGENT] Ticket {state['ticket_id']} requires manual review.",
        "resolution_path": "escalated",
        "escalation_reason": reason
    }


print("All nodes defined: classify → auto_resolve / draft_response / escalate_to_human")

## Step 4 — Build the Graph with Conditional Routing

In [ ]:
def route_by_difficulty(state: SupportState) -> str:
    """Route to the appropriate handler based on difficulty."""
    difficulty = state.get("difficulty", "medium").strip().lower()
    if "easy" in difficulty:
        return "auto_resolve"
    elif "hard" in difficulty:
        return "escalate_to_human"
    else:
        return "draft_response"


# Build graph
workflow = StateGraph(SupportState)

workflow.add_node("classify_ticket", classify_ticket)
workflow.add_node("auto_resolve", auto_resolve)
workflow.add_node("draft_response", draft_response)
workflow.add_node("escalate_to_human", escalate_to_human)

workflow.set_entry_point("classify_ticket")

# Conditional routing based on difficulty
workflow.add_conditional_edges(
    "classify_ticket",
    route_by_difficulty,
    {
        "auto_resolve": "auto_resolve",
        "draft_response": "draft_response",
        "escalate_to_human": "escalate_to_human",
    }
)

# All resolution paths end
workflow.add_edge("auto_resolve", END)
workflow.add_edge("draft_response", END)
workflow.add_edge("escalate_to_human", END)

app = workflow.compile()
print("Support workflow compiled with 3-way routing")

## Step 5 — Process All Tickets

In [ ]:
results = []

for ticket in tickets:
    print(f"\n{'='*60}")
    print(f"📨 Ticket {ticket['ticket_id']} from {ticket['customer_name']}")
    print(f"   {ticket['ticket_text'][:80]}..." if len(ticket['ticket_text']) > 80 else f"   {ticket['ticket_text']}")

    state = {
        "ticket_id": ticket["ticket_id"],
        "customer_name": ticket["customer_name"],
        "ticket_text": ticket["ticket_text"],
        "category": "",
        "difficulty": "",
        "sentiment": "",
        "response": "",
        "resolution_path": "",
        "escalation_reason": "",
    }

    result = app.invoke(state)
    results.append(result)

    print(f"\n  📌 Path: {result['resolution_path']}")
    print(f"  📩 Response: {result['response'][:120]}..." if len(result['response']) > 120 else f"  📩 Response: {result['response']}")

## Step 6 — Summary Dashboard

In [ ]:
print("\n" + "="*60)
print("📊 SUPPORT ROUTING SUMMARY")
print("="*60)

path_counts = {}
for r in results:
    path = r["resolution_path"]
    path_counts[path] = path_counts.get(path, 0) + 1

print(f"\nTotal tickets: {len(results)}")
for path, count in sorted(path_counts.items()):
    emoji = {"auto_resolved": "✅", "drafted": "📝", "escalated": "🚨"}.get(path, "❓")
    print(f"  {emoji} {path}: {count}")

print(f"\n{'ID':<8} {'Customer':<16} {'Category':<12} {'Difficulty':<10} {'Path':<15}")
print("-" * 60)
for r in results:
    print(f"{r['ticket_id']:<8} {r['customer_name']:<16} {r['category']:<12} {r['difficulty']:<10} {r['resolution_path']:<15}")

# Show escalation details
escalated = [r for r in results if r["resolution_path"] == "escalated"]
if escalated:
    print(f"\n🚨 Escalated Tickets Detail:")
    for r in escalated:
        print(f"\n  Ticket: {r['ticket_id']} ({r['customer_name']})")
        print(f"  Reason: {r['escalation_reason'][:200]}...")

## 🧠 Key Concepts Recap

| LangGraph Feature | What It Does | Used Here |
|-------------------|--------------|-----------|
| `add_conditional_edges()` | Route to different nodes based on state | Difficulty-based routing |
| Route function | Returns the name of the target node | `route_by_difficulty()` |
| Route map | Maps return values to node names | `{"auto_resolve": ..., ...}` |
| Multiple END edges | Different branches can all lead to END | All 3 paths → END |

## Routing Patterns

```python
# Simple: return node name directly
def router(state) -> str:
    if state["x"]: return "node_a"
    return "node_b"

# With explicit map
graph.add_conditional_edges("classifier", router, {
    "node_a": "node_a",
    "node_b": "node_b",
})
```

## 🔧 Extensions

- **Priority queue**: High-severity escalations get processed first
- **SLA tracking**: Add timestamps and alert on SLA breaches
- **Feedback loop**: If customer replies "not helpful", re-route to higher tier
- **Knowledge base**: Auto-resolve checks a FAQ database before generating